# Published EACF on real Kepler and TESS targets

This notebook applies Urdr's published EACF workflow to two established solar-like oscillators downloaded from MAST through Mimir:

- **KIC 8006161 (HD 173701):** one Kepler short-cadence quarter, illustrating a high-frequency main-sequence oscillator.
- **HD 212771 (TIC 12723961):** one TESS 2-minute sector, illustrating a low-frequency RGB oscillator.

For each target we retain the real timestamps and gaps, calculate a Mimir power-density spectrum, compare Urdr's broad scaling-relation search with an AsteroScale-informed search, and display both the raw and exact-window null-standardized EACF maps.

The notebook deliberately downloads one observing segment per target. Increase `products` only after the small example works locally. MAST access requires the `real-data` extra:

```bash
python -m pip install -e ".[asteroscale,real-data]"
```


In [ ]:
from pathlib import Path

import asteroscale as ast
import matplotlib.pyplot as plt
import numpy as np

from mimir import (
    download_lightcurves,
    lightcurve_to_timeseries,
    power_spectrum,
    reduce_lightcurve,
    search_lightcurves,
)
from urdr import (
    AsteroScaleSamples,
    EmpiricalBackgroundConfig,
    PublishedEACFSearch,
    SimulationConfig,
    TimeSeries,
    calibrate_published_eacf,
    compute_published_eacf_map,
    envelope_width_uhz,
    estimate_background,
)

DOWNLOAD_DIR = Path("data/mast")
SIMULATIONS = 64       # Use >=1000 for a resolved small global p_FA.
BATCH_SIZE = 8         # Reduce this if a long Kepler segment exhausts memory.
FFT_WORKERS = 2
SEED = 20260802


## 1. Target definitions

The literature seismic values are used only as reference markers and to choose a practical plotting range. They are not passed to the broad EACF statistic. Campante et al. (2019) report $\nu_{\max}=226.6\pm9.4\,\mu$Hz and $\Delta\nu=16.25\pm0.19\,\mu$Hz for HD 212771.

The stellar inputs are passed to AsteroScale to construct the explicitly prior-informed search. The two detection products remain separate.


In [ ]:
TARGETS = {
    "Kepler — KIC 8006161": {
        "target": "KIC 8006161",
        "search_kwargs": {"mission": "Kepler", "author": "Kepler", "exptime": 60},
        "products": 1,
        "literature_numax": 3575.0,
        "literature_dnu": 149.4,
        "stellar": {
            "M": (1.00, 0.03),
            "R": (0.93, 0.01),
            "Teff": (5488.0, 77.0),
            "FeH": (0.30, 0.10),
        },
    },
    "TESS — HD 212771": {
        "target": "TIC 12723961",
        "search_kwargs": {"mission": "TESS", "author": "SPOC", "exptime": 120},
        "products": 1,
        "literature_numax": 226.6,
        "literature_dnu": 16.25,
        "stellar": {
            "M": (1.45, 0.10),
            "R": (4.45, 0.10),
            "Teff": (5065.0, 75.0),
            "FeH": (-0.10, 0.10),
        },
    },
}


## 2. Download with Mimir and preserve the observing window

Mimir returns only valid timestamps. Urdr's null simulator instead needs a regular cadence grid plus an explicit observing mask. `to_urdr_series` maps the observed points onto that grid without filling the gaps with zero flux.


In [ ]:
def load_one_segment(settings):
    result = search_lightcurves(settings["target"], **settings["search_kwargs"])
    selected = result[: settings["products"]]
    collection = download_lightcurves(selected, download_dir=DOWNLOAD_DIR)
    reduced = reduce_lightcurve(
        collection,
        outlier_sigma=5.0,
        flatten=True,
        exposure_time=settings["search_kwargs"]["exptime"],
        numax=settings["literature_numax"],
    )
    return lightcurve_to_timeseries(reduced, ppm=True)


def to_urdr_series(mimir_series):
    time = np.asarray(mimir_series.time, dtype=float)
    flux = np.asarray(mimir_series.flux, dtype=float)
    finite = np.isfinite(time) & np.isfinite(flux)
    time, flux = time[finite], flux[finite]

    cadence = float(np.median(np.diff(time)))
    indices = np.rint((time - time[0]) / cadence).astype(int)
    unique = np.concatenate(([True], np.diff(indices) > 0))
    indices, flux = indices[unique], flux[unique]

    grid = time[0] + cadence * np.arange(indices[-1] + 1)
    gridded_flux = np.full(grid.size, np.nan)
    gridded_flux[indices] = flux
    return TimeSeries.from_arrays(grid, gridded_flux)


series = {}
for label, settings in TARGETS.items():
    downloaded = load_one_segment(settings)
    series[label] = to_urdr_series(downloaded)
    item = series[label]
    print(
        f"{label}: {item.time[-1] - item.time[0]:.1f} d, "
        f"{item.cadence_seconds:.1f} s cadence, "
        f"{item.window.duty_cycle:.1%} duty cycle"
    )


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), constrained_layout=True)
for ax, (label, item) in zip(axes, series.items()):
    ax.plot(item.time[item.observed] - item.time[0], item.flux[item.observed], lw=0.35)
    ax.set(title=label, ylabel="Relative flux [ppm]")
axes[-1].set_xlabel("Time since first cadence [d]")
plt.show()


## 3. Mimir spectra and target-aware empirical backgrounds

The oscillation envelope is excluded from the running-median background estimate. This is important for high-S/N targets: otherwise the modes can lift the fitted background and suppress the centre of the envelope after whitening.


In [ ]:
spectra = {}
backgrounds = {}

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, (label, item) in zip(axes, series.items()):
    settings = TARGETS[label]
    observed = item.observed
    spectrum = power_spectrum(
        time=item.time[observed],
        flux=item.flux[observed],
        time_unit="d",
        frequency_unit="uHz",
    )
    width = float(envelope_width_uhz(settings["literature_numax"]))
    background_config = EmpiricalBackgroundConfig.excluding_envelope(
        settings["literature_numax"], width
    )
    background = estimate_background(
        spectrum.frequency, spectrum.power_density, background_config
    )
    spectra[label] = spectrum
    backgrounds[label] = background_config

    lower = 0.35 * settings["literature_numax"]
    upper = 1.65 * settings["literature_numax"]
    visible = (spectrum.frequency >= lower) & (spectrum.frequency <= upper)
    ax.plot(spectrum.frequency[visible], spectrum.power_density[visible], color="0.65", lw=0.5)
    ax.plot(spectrum.frequency[visible], background[visible], color="C1", lw=2)
    ax.axvline(settings["literature_numax"], color="C3", ls="--", label="literature numax")
    ax.set(title=label, xlabel="Frequency [uHz]", ylabel="Power density")
    ax.set_yscale("log")
    ax.legend()
plt.show()


## 4. Broad published-EACF maps

These maps use a deliberately broad set of filter centres around the visible excess and Urdr's default $\nu_{\max}$--$\Delta\nu$ banana. The literature marker is overplotted only after the map is calculated.


In [ ]:
broad_maps = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

for ax, (label, item) in zip(axes, series.items()):
    settings = TARGETS[label]
    centres = np.linspace(
        0.55 * settings["literature_numax"],
        1.45 * settings["literature_numax"],
        51,
    )
    result = compute_published_eacf_map(
        item,
        centres,
        background=backgrounds[label],
    )
    broad_maps[label] = result
    masked = np.where(result.physical_mask, result.values, np.nan)
    image = ax.pcolormesh(
        result.centre_frequencies_uhz,
        result.lags_seconds / 3600.0,
        masked.T,
        shading="auto",
    )
    ax.scatter(
        [settings["literature_numax"]],
        [1e6 / settings["literature_dnu"] / 3600.0],
        marker="x",
        s=70,
        color="red",
        label="literature",
    )
    ax.set(title=label, xlabel="Trial filter centre [uHz]", ylabel="Lag [h]")
    ax.legend()
    fig.colorbar(image, ax=ax, label="Raw normalized EACF")
plt.show()


## 5. AsteroScale-informed searches

AsteroScale supplies trial centres, conditional $\Delta\nu$ ranges, and envelope widths. This is a separate conditional analysis, not additional evidence extracted from the light curve.


In [ ]:
informed_searches = {}

for offset, (label, settings) in enumerate(TARGETS.items()):
    prediction = ast.solve(
        given=settings["stellar"],
        want=["numax", "dnu", "FWHM_env"],
        preset="fast",
        seed=SEED + offset,
    )
    samples = AsteroScaleSamples.from_mapping(prediction)
    informed_searches[label] = PublishedEACFSearch.from_asteroscale(
        samples,
        centre_count=21,
        credible_mass=0.99,
    )
    print(label, samples.median_parameters())


## 6. Exact-window null calibration

The helper below estimates a white-noise scale from first differences and assigns the remaining robust variance to a simple Ornstein--Uhlenbeck granulation component. This is an intentionally lightweight starting point for a real-data tutorial. It reproduces the real window exactly, but it is **not** a substitute for fitting a scientifically adequate granulation/background model.

With 64 nulls, the smallest possible global false-alarm probability is $1/65$. Increase `SIMULATIONS` to at least 1000 before interpreting a small $p_{\rm FA}$, and validate the null model independently.


In [ ]:
def robust_scale(values):
    values = np.asarray(values, dtype=float)
    centre = np.median(values)
    return 1.4826 * np.median(np.abs(values - centre))


def empirical_null_config(item, settings):
    observed_flux = item.flux[item.observed]
    white = robust_scale(np.diff(observed_flux)) / np.sqrt(2.0)
    total = robust_scale(observed_flux)
    granulation = np.sqrt(max(total**2 - white**2, (0.1 * white) ** 2))
    timescale_days = 1.0 / (
        2.0 * np.pi * settings["literature_numax"] * 1e-6 * 86400.0
    )
    return SimulationConfig(
        white_noise_sigma=white,
        granulation_amplitude=granulation,
        granulation_timescale_days=timescale_days,
        numax_uhz=settings["literature_numax"],
        delta_nu_uhz=settings["literature_dnu"],
        envelope_width_uhz=float(envelope_width_uhz(settings["literature_numax"])),
        oscillation_amplitude=0.0,
    )


detections = {}
for offset, (label, item) in enumerate(series.items()):
    settings = TARGETS[label]
    simulation = empirical_null_config(item, settings)
    broad = calibrate_published_eacf(
        item,
        simulation,
        broad_maps[label].centre_frequencies_uhz,
        background=backgrounds[label],
        simulations=SIMULATIONS,
        batch_size=BATCH_SIZE,
        fft_workers=FFT_WORKERS,
        seed=SEED + 10 * offset,
    )
    informed = calibrate_published_eacf(
        item,
        simulation,
        search=informed_searches[label],
        background=backgrounds[label],
        simulations=SIMULATIONS,
        batch_size=BATCH_SIZE,
        fft_workers=FFT_WORKERS,
        seed=SEED + 10 * offset,
    )
    detections[label] = {"broad": broad, "informed": informed}
    print(
        f"{label}\n"
        f"  broad:   p_FA={broad.false_alarm_probability:.4f}, "
        f"numax={broad.window_aware_best_numax_uhz:.1f} uHz, "
        f"dnu={broad.window_aware_best_delta_nu_uhz:.2f} uHz\n"
        f"  informed: p_FA={informed.false_alarm_probability:.4f}, "
        f"numax={informed.window_aware_best_numax_uhz:.1f} uHz, "
        f"dnu={informed.window_aware_best_delta_nu_uhz:.2f} uHz"
    )


## 7. Raw and null-standardized localisation

The raw map shows the actual EACF, including recurring filter/window structure. The standardized residual

$$Z(\nu,\tau)=\frac{E_{\rm obs}-\mu_{\rm null}}{\sigma_{\rm null}}$$

suppresses structure reproduced by exact-window null simulations. Its collapse is useful for localisation; the globally calibrated `false_alarm_probability`, not a pointwise $Z$, is the detection result.


In [ ]:
for label, pair in detections.items():
    settings = TARGETS[label]
    result = pair["informed"]
    observed = result.observed
    raw = np.where(observed.physical_mask, observed.values, np.nan)
    standardized = np.where(
        observed.physical_mask, result.null_standardized_score, np.nan
    )

    finite = standardized[np.isfinite(standardized)]
    z_limit = max(1.0, float(np.nanpercentile(np.abs(finite), 99)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    raw_image = axes[0].pcolormesh(
        observed.centre_frequencies_uhz,
        observed.lags_seconds / 3600.0,
        raw.T,
        shading="auto",
    )
    z_image = axes[1].pcolormesh(
        observed.centre_frequencies_uhz,
        observed.lags_seconds / 3600.0,
        standardized.T,
        shading="auto",
        cmap="coolwarm",
        vmin=-z_limit,
        vmax=z_limit,
    )
    axes[2].plot(
        observed.centre_frequencies_uhz,
        result.null_standardized_collapsed(),
        label="informed standardized collapse",
    )
    axes[2].axvline(
        settings["literature_numax"], color="C3", ls="--", label="literature"
    )
    for ax in axes[:2]:
        ax.scatter(
            [settings["literature_numax"]],
            [1e6 / settings["literature_dnu"] / 3600.0],
            marker="x",
            s=70,
            color="black",
        )
        ax.set(xlabel="Trial filter centre [uHz]", ylabel="Lag [h]")
    axes[0].set_title("Raw EACF")
    axes[1].set_title("Exact-window null-standardized EACF")
    axes[2].set(
        title="Standardized collapse",
        xlabel="Trial filter centre [uHz]",
        ylabel="Mean Z inside allowed region",
    )
    axes[2].legend()
    fig.colorbar(raw_image, ax=axes[0], label="Raw normalized EACF")
    fig.colorbar(z_image, ax=axes[1], label="Null-standardized residual Z")
    fig.suptitle(label)
    plt.show()


## 8. Interpretation checklist

For each star, compare:

1. the visible power excess and empirical background;
2. the raw EACF ridge and its $1/\Delta\nu$ repetition;
3. structures that disappear after exact-window null standardization;
4. broad versus AsteroScale-informed $p_{\rm FA}$ values, keeping their different search volumes explicit;
5. recovered values against the literature markers, remembering that the winning Hanning centre is a localisation estimate rather than a precision measurement of $\nu_{\max}$.

Before using the false-alarm probabilities scientifically, increase the null count, fit or externally constrain the granulation model, repeat the analysis across observing segments, and run signal-injection recovery tests through the same Mimir reduction and exact window.

References: Appourchaux et al. (2012), A&A 543, A54; Campante et al. (2019), ApJ 885, 31.
